# Import Module

In [9]:
import json
from urllib.request import urlopen, Request
from urllib.parse import quote
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Load data from extracted database

In [10]:
# Load data
with open("extracted_database.json", "r") as f:
    data = json.load(f)

# User Input Keyword

In [11]:
input = "angina"

# Generate Related Terms

In [14]:
# SNOMED API configuration for related terms
baseUrl = 'https://browser.ihtsdotools.org/snowstorm/snomed-ct'
edition = 'MAIN'
version = '2024-10-01'
user_agent = 'manucianhh@gmail.com'

def urlopen_with_header(url):
    req = Request(url)
    req.add_header('User-Agent', user_agent)
    return urlopen(req)

def getSnomedCode(searchTerm):
    url = f"{baseUrl}/browser/{edition}/{version}/descriptions?term={quote(searchTerm)}&conceptActive=true&groupByConcept=false&searchMode=STANDARD&offset=0&limit=50"
    response = urlopen_with_header(url).read()
    data = json.loads(response.decode('utf-8'))

    for term in data['items']:
        if searchTerm.lower() == term['term'].lower():
            return term['concept']['conceptId']

def getConceptById(concept_id):
    url = f"{baseUrl}/browser/{edition}/{version}/concepts/{concept_id}"
    response = urlopen_with_header(url).read()
    return json.loads(response.decode('utf-8'))

def getRelatedTerms(concept):
    terms = []
    for item in concept['descriptions']:
        if item['active']:
            terms.append(item['term'])
    return terms

def finetuneTerms(originalTerm, relatedTerms):
    terms = []
    for term in relatedTerms:
        if (originalTerm.lower() != term.lower()) and  ("(" not in term):
            terms.append(term)
    return terms

In [15]:
# Generate and Fine-tune Related Terms
originalTerm = input
id = getSnomedCode(originalTerm)
concept = getConceptById(id)
relatedTerms = getRelatedTerms(concept)
relatedTerms = finetuneTerms(originalTerm, relatedTerms)

# Select Candidates

In [16]:
# Select Candidates
candidates = []
for item in data:
    if (item["definition"] != "Definition not found"):
        if (originalTerm.lower().strip() in item["term"].lower().strip()):
            candidates.append(item)
        else:
            for term in relatedTerms:
                if (term.lower().strip() in item["term"].lower().strip()):
                    candidates.append(item)
                    break

# Ranking by Cosine Similarity

In [17]:
# Extract definitions for similarity computation
definitions = [entry["definition"] for entry in candidates]

# Create a TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the medical terminology and definitions into TF-IDF vectors
tfidf_matrix = vectorizer.fit_transform([originalTerm] + definitions)

# Calculate cosine similarity between the medical term and each definition
similarities = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()

# Add cosine similarity scores to the dataset and rank
for i, entry in enumerate(candidates):
    entry["cosine_similarity"] = similarities[i]

# Sort data by similarity
ranked_results = sorted(candidates, key=lambda x: x["cosine_similarity"], reverse=True)

# Add numerical order column
for index, entry in enumerate(ranked_results, start=1):
    entry["rank"] = index

# Convert to pandas DataFrame for better display
print("\033[1m" + "Orginal Medical Terminology: " + "\033[0;0m", originalTerm)
print("\033[1m" + "List of Related Temrinologys: " + "\033[0;0m", relatedTerms)
df = pd.DataFrame(ranked_results, columns=["rank", "definition", "symptoms", "diagnosis", "treatment", "risk_factors", "cosine_similarity"])
pd.set_option('display.max_colwidth', None)
df.head(20)

Orginal Medical Terminology:  angina
List of Related Temrinologys:  ['Stenocardia', 'Anginal syndrome', 'Angina pectoris', 'Ischemic heart disease - angina', 'AP - Angina pectoris', 'Cardiac angina', 'Ischaemic heart disease - angina']


rank  \
0     1   
1     2   
2     3   
3     4   
4     5   
5     6   
6     7   
7     8   
8     9   
9    10   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         